In [0]:

# Constantes del laboratorio — usar en todos los notebooks de tareas
CATALOG       = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing"
SOURCE_PATH   = f"{VOLUME_PATH}/sesion09"

BRONZE_TABLE  = f"{CATALOG}.{SCHEMA_BRONZE}.transacciones_retail_raw"
SILVER_TABLE  = f"{CATALOG}.{SCHEMA_SILVER}.transacciones_clean"
GOLD_TABLE    = f"{CATALOG}.{SCHEMA_GOLD}.ventas_por_tienda"

print(f"Source  : {SOURCE_PATH}")
print(f"Bronze  : {BRONZE_TABLE}")
print(f"Silver  : {SILVER_TABLE}")
print(f"Gold    : {GOLD_TABLE}")

In [0]:

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.functions import current_timestamp, col, lit

schema_transacciones = StructType([
    StructField("order_id",      StringType(),  False),
    StructField("fecha",         DateType(),    True),
    StructField("tienda_id",     StringType(),  True),
    StructField("cliente_id",    StringType(),  True),
    StructField("producto",      StringType(),  True),
    StructField("categoria",     StringType(),  True),
    StructField("cantidad",      IntegerType(), True),
    StructField("precio_unit",   DoubleType(),  True),
    StructField("descuento_pct", DoubleType(),  True),
    StructField("total",         DoubleType(),  True),
    StructField("metodo_pago",   StringType(),  True),
    StructField("estado",        StringType(),  True),
])

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema_transacciones)
    .load(f"{SOURCE_PATH}/transacciones_retail.csv")
)

# Columnas técnicas de auditoría Bronze
df_bronze = (
    df_raw
    .withColumn("_ingestion_ts",  current_timestamp())
    .withColumn("_source_file",   col("_metadata.file_name"))
    .withColumn("_batch_session", lit("sesion09"))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze cargado: {bronze_count} registros -> {BRONZE_TABLE}")

# Task Value: disponible para la tarea silver_transform en el Workflow
dbutils.jobs.taskValues.set(key="bronze_count", value=bronze_count)